In [161]:
import pickle
from idlelib.iomenu import encoding

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split

import wandb

In [162]:
wandb.login()

True

In [189]:
def get_subject_samples(data):
    chest_signals = data['signal']['chest']

    c_acc = chest_signals['ACC']
    c_acc_np = np.array(c_acc)
    acc_mag = np.sum(c_acc_np**2, axis=1)

    acc = acc_mag.reshape(-1, 1)
    ecg = chest_signals['ECG']
    emg = chest_signals['EMG']
    eda = chest_signals['EDA']
    temp = chest_signals['Temp']
    resp = chest_signals['Resp']

    return torch.tensor(np.stack([acc, ecg, emg, eda, temp, resp], axis=1).squeeze()).float()

def get_subject_targets(data):
    targets = data['label']
    return torch.tensor(targets.reshape(-1, 1)).long().squeeze(dim=1)

def normalize_samples(samples: torch.Tensor, test_samples: torch.Tensor = None):
    mean = samples.mean(dim=0, keepdim=True)
    std = samples.std(dim=0, keepdim=True)

    samples = (samples - mean) / std

    if test_samples != None:
        test_samples = (test_samples - mean) / std
        return samples, test_samples

    return samples

def get_model_accuracy(logits, targets):
    probs = torch.softmax(logits, dim=1)
    ind = torch.argmax(probs, dim=1)

    diff = (ind == targets).int()
    correct = torch.sum(diff, dim=0).item()
    accuracy =  float(format((correct / len(logits)) * 100, ".2f"))
    return accuracy

def train_subject_model(x_test_p, y_test_p, m, optim, epochs, data_loader, run, subject):
    for e in tqdm(range(epochs), desc='Epochs'):
        for x, y in tqdm(data_loader, desc='Batch'):
            m.train()
            pred = m(x)
            loss = loss_fn(pred, y)
            run.log({
                f"{subject}_training_loss":loss.item()
            })

            optim.zero_grad()
            loss.backward()
            optim.step()

        model.eval()
        with torch.no_grad():
            #pred_logits = m(x_test_p)
            # pred_mx = torch.softmax(pred_logits, dim=1)
            # pred_ind = torch.argmax(pred_mx, dim=1)
            #
            # correct = (pred_ind == y_test_p).int()
            # num_correct = torch.sum(correct, dim=0).item()
            # accuracy =  float(format((num_correct / len(y_test_p)) * 100, ".2f"))
            pred_logits = m(x_test_p)
            accuracy = get_model_accuracy(pred_logits, y_test_p)

            print(f"accuracy {accuracy}%")
            run.log({
                f"{subject}_accuracy":accuracy
            })

    return model.state_dict()

def print_signal(data):
    keys = data.keys()
    for k in keys:
        print(f"{k} has length of {len(data[k])}")

def test_sample(data):
    sample = []
    for k in data.keys():
        record = data[k]
        print(record.shape)
        sample.append(record[0])
    print(sample)

class AffectModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(in_features=6, out_features=64),
            nn.ReLU(),
            nn.Linear(in_features=64, out_features=128),
            nn.ReLU(),
            nn.Linear(in_features=128, out_features=64),
            nn.ReLU(),
            nn.Linear(in_features=64, out_features=8),
        )

    def forward(self, x):
        return self.model(x)


In [ ]:
def train_federated():
    # So 1 round of federated learning is training all local models for e number of epochs
    # I then take the weighted average of each layer for each model add them together and then test..that's one round

    for n in num_rounds:
        clients = dict()
        for s in subjects:
            dl = get_data_loader(s, c_batch_size)
            c_m = train(c_epochs, dl)
            clients[s] = {model: c_m.state_dict(), num_samples: len(dl)}

        agg_model = fed_avg(clients)
        agg_accuray = fed_avg_accuracy(agg_model)




In [169]:
#subjects = ["S6","S7","S8","S9"]
subjects = ["S7","S8","S9"]
all_models = []

c = 3 # number of clients that participate in each round
c_epochs = 5 #Number of training passes each client makes over local dataset
c_batch_size = 250

In [170]:
BATCH_SIZE = 256
MAX_EPOCH = 10
LEARNING_RATE = 1e-3
PROJECT_PREFIX = "stress-fl"
rng = np.random.default_rng()

run = wandb.init(
    entity='hutchtech',
    project="stress-fl",
    name=f"{PROJECT_PREFIX}-{rng.random()}",
    config={
        "lr":LEARNING_RATE,
        "epochs":MAX_EPOCH,
        "batch_size":BATCH_SIZE
    }
)

wandb: Tracking run with wandb version 0.28.1
wandb: Run data is saved locally in /Users/hutchii/projects/MentalHealthLLM/notebooks/wandb/run-20260808_205637-58gi9fml
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run stress-fl-0.9871860900620284
wandb: ⭐️ View project at https://wandb.ai/hutchtech/stress-fl
wandb: 🚀 View run at https://wandb.ai/hutchtech/stress-fl/runs/58gi9fml


In [171]:
for s in subjects:
    with open(f"/Users/hutchii/projects/MentalHealthLLM/data/wesad_subjects/{s}.pkl", 'rb') as f:
        data = pickle.load(f, encoding='latin1')

    s_samples = get_subject_samples(data)
    s_targets = get_subject_targets(data)

    print(s_samples.shape)
    print(s_targets.shape)

    x_train, x_test, y_train, y_test = train_test_split(
        s_samples,
        s_targets,
        test_size=0.25,
        stratify=s_targets,
        shuffle=True,
    )

    x_train_norm, x_test_norm = normalize_samples(x_train, x_test)

    #Dataset
    ds = TensorDataset(x_train_norm, y_train)
    dl = DataLoader(dataset=ds, shuffle=True, batch_size=BATCH_SIZE)

    #model
    model = AffectModel()
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    loss_fn = nn.CrossEntropyLoss()

    trained_model = train_subject_model(
        subject = s,
        run=run,
        x_test_p=x_test_norm,
        y_test_p=y_test,
        data_loader=dl,
        m=model,
        optim=optimizer,
        epochs=MAX_EPOCH
    )

    all_models.append(trained_model)

torch.Size([3666600, 6])
torch.Size([3666600])


Epochs:   0%|          | 0/10 [00:00<?, ?it/s]

Batch:   0%|          | 0/10742 [00:00<?, ?it/s]

accuracy 92.28%


Batch:   0%|          | 0/10742 [00:00<?, ?it/s]

...

accuracy 94.52%


Batch:   0%|          | 0/10712 [00:00<?, ?it/s]

accuracy 94.6%


Batch:   0%|          | 0/10712 [00:00<?, ?it/s]

accuracy 94.61%


In [192]:
s7_len = 3666600
s8_len = 3826200
s9_len = 3656100
total_samples = s7_len + s8_len + s9_len

keys = all_models[0].keys()

layers = dict()
for k in keys:
    sample_model = all_models[0]
    shape = sample_model[k].shape
    weighted_sum = torch.zeros(shape)
    for i, m in enumerate(all_models):
        subject = s7_len if i == 0 else (s8_len if i == 1 else s9_len)
        weight = subject / total_samples
        weighted_sum += weight * m[k]
    layers[k] = weighted_sum

print(layers.keys())

dict_keys(['model.0.weight', 'model.0.bias', 'model.2.weight', 'model.2.bias', 'model.4.weight', 'model.4.bias', 'model.6.weight', 'model.6.bias'])


In [193]:
agg_model = AffectModel()
agg_model.load_state_dict(layers)
agg_model.eval()

for s in subjects:
    with open(f"/Users/hutchii/projects/MentalHealthLLM/data/wesad_subjects/{s}.pkl", 'rb') as f:
        data = pickle.load(f, encoding='latin1')

    s_samples = get_subject_samples(data)
    s_targets = get_subject_targets(data)

    x_train_norm = normalize_samples(s_samples)

    with torch.no_grad():
        pred_l = agg_model(x_train_norm)
        accuracy = get_model_accuracy(pred_l, s_targets)
        print(f"{s} accuracy is {accuracy}")

S7 accuracy is 28.49
S8 accuracy is 23.78
S9 accuracy is 27.87


In [173]:
s9_m = all_models[0]
print(s9_m.keys())

# for k,v in s9_m.items():
#     print(k, v.shape)

odict_keys(['model.0.weight', 'model.0.bias', 'model.2.weight', 'model.2.bias', 'model.4.weight', 'model.4.bias', 'model.6.weight', 'model.6.bias'])


In [159]:
baseline_run.finish()